In [48]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from go_ml.eval_utils import filter_annot_df
from go_ml.eval_utils import (load_msa_dict, gen_bert_mat, get_bert_entropy, 
                              gen_annot_mat, gen_seq_len_mask, mean_reciprocal_rank, 
                              mean_reciprocal_rank_mat, mean_auc, top_30_score, roc_average)

data_root = '../gen_datasets/datasets'
csa_df = filter_annot_df(pd.read_csv(f'{data_root}/csa_annot.csv', sep='\t'))
llps_df = filter_annot_df(pd.read_csv(f'{data_root}/llps_dataset.csv', sep='\t'))
elms_df = filter_annot_df(pd.read_csv(f'{data_root}/elms_dataset.csv', sep='\t'))

df_n = ['csa', 'llps', 'elms']
df_l = [csa_df, llps_df, elms_df]

# np.random.seed(42)
# train_row_mask = np.random.rand(len(annot_df)) > 0.5
# train_df = annot_df[train_row_mask]
# val_df = annot_df[~train_row_mask]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from go_ml.models.esm_token_finetune import ESMCTokenFinetune
# (func_cond_finetune/version_2) Func conditioned model, normal bert masking
# (func_cond_finetune/version_4) Func conditioned model, crazy span masking
#model_path is the path matching f"../../checkpoints/annot_esm_warmup-{df_name}-v{k}.ckpt" with highest k
for ind in range(3):
    annot_df = df_l[ind]
    ds_label = df_n[ind]

    import os
    root_path = '/home/andrew/GO_interp/checkpoints/'
    model_paths = os.listdir(root_path)
    model_path = ([f for f in model_paths if f.startswith(f"annot_esm_warmup-perc0.1-{ds_label}.ckpt")] + 
                sorted([f for f in model_paths if f.startswith(f"annot_esm_warmup-perc0.1-{ds_label}-v")]))[-1]
    import json
    train_path = "/home/andrew/cafa5_team/data/"
    with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
        go_terms = json.load(f)
    print(f'Loading model from {model_path}')
    device = torch.device('cuda:0')
    model = ESMCTokenFinetune.load_from_checkpoint(f"{root_path}/{model_path}", map_location=device)
    model = model.eval()

    from go_ml.data_utils import ProtFuncDataset, prot_func_collate, dict_to_device
    from tqdm.notebook import tqdm
    annot_ds = ProtFuncDataset.from_annot_df(annot_df, go_terms=go_terms)
    val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn": prot_func_collate}
    prot_ids = []
    out_l = []
    with torch.no_grad():
        # for batch in torch.utils.data.DataLoader(annot_ds, **val_dataloader_params):
        for batch in tqdm(torch.utils.data.DataLoader(annot_ds, **val_dataloader_params), total=len(annot_ds)//val_dataloader_params['batch_size']):
            dict_to_device(batch, device)
            prot_ids.extend(batch['prot_id'])
            seq_ind, mask, annot_labels = (batch['seq_ind'], batch['mask'], batch['annot_mask'])
            out = model.forward(seq_ind, mask)
            out_l.append(out.logits.cpu())
            
    from itertools import chain
    score_l = list(chain.from_iterable([m[i] for i in range(m.shape[0])] for m in out_l))
    score_map = {pid: score for pid, score in zip(prot_ids, score_l)}
    from go_ml.eval_utils import gen_logit_map
    model_logits = gen_logit_map(prot_ids, score_map, max_len=850)
    model_pred = torch.argmax(torch.tensor(model_logits), dim=-1)
    model_scores = torch.softmax(torch.tensor(model_logits), dim=-1)[..., 1]

    import pickle
    with open(f'eval_files/{ds_label}_score_pred.pkl', 'wb') as f:
        pickle.dump({'UniprotID': annot_df['UniprotID'], 'score_mat': model_scores.numpy()}, f)

Loading model from annot_esm_warmup-perc0.1-csa.ckpt


Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/65 [00:00<?, ?it/s]

Loading model from annot_esm_warmup-perc0.1-llps.ckpt


Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/4 [00:00<?, ?it/s]

Loading model from annot_esm_warmup-perc0.1-elms.ckpt


Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/19 [00:00<?, ?it/s]

In [47]:
from go_ml.data_utils import ProtFuncDataset, prot_func_collate, dict_to_device
from tqdm.notebook import tqdm
annot_ds = ProtFuncDataset.from_annot_df(annot_df, go_terms=go_terms)
val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn": prot_func_collate}
prot_ids = []
out_l = []
with torch.no_grad():
    # for batch in torch.utils.data.DataLoader(annot_ds, **val_dataloader_params):
    for batch in tqdm(torch.utils.data.DataLoader(annot_ds, **val_dataloader_params), total=len(annot_ds)//val_dataloader_params['batch_size']):
        dict_to_device(batch, device)
        prot_ids.extend(batch['prot_id'])
        seq_ind, mask, annot_labels = (batch['seq_ind'], batch['mask'], batch['annot_mask'])
        out = model.forward(seq_ind, mask)
        out_l.append(out.logits.cpu())
        
from itertools import chain
score_l = list(chain.from_iterable([m[i] for i in range(m.shape[0])] for m in out_l))
score_map = {pid: score for pid, score in zip(prot_ids, score_l)}
from go_ml.eval_utils import gen_logit_map
model_logits = gen_logit_map(prot_ids, score_map, max_len=850)
model_pred = torch.argmax(torch.tensor(model_logits), dim=-1)
model_scores = torch.softmax(torch.tensor(model_logits), dim=-1)[..., 1]

import pickle
with open(f'eval_files/{ds_label}_score_pred.pkl', 'wb') as f:
    pickle.dump({'UniprotID': annot_df['UniprotID'], 'score_mat': model_scores.numpy()}, f)

  0%|          | 0/19 [00:00<?, ?it/s]